# SN-03 — Sirenisation Phase 2 (top 5 probabiliste)

Sur les EJ du périmètre A/B/C, recherche des **5 meilleures UL candidates** par EJ. Blocking sur le code commune du siège UL.

Statuts : VALIDE_FORT / VALIDE / DOUTEUX / REJETE / SANS_CANDIDAT

In [1]:
import sys, os
sys.path.insert(0, os.path.abspath('../..'))

import pandas as pd
from tqdm.auto import tqdm

from src.sirenisation import scorer_paire_ej_ul
from src.matching     import classifier_resultat
from src.excel_export import export_topn_excel
from src.display      import afficher_tableau, afficher_synthese
from config.settings  import (
    SN_PERIMETRE, SIRENE_UL_CLEAN, SN_PHASE1, SN_PHASE2, RESULTS_SN_DIR,
)
RESULTS_SN_DIR.mkdir(parents=True, exist_ok=True)

# Récupération des SIREN validés en P1 (pour désactiver le bonus sur les EJ jumeaux)
FEUILLES_VALIDES = {'Valide_fort', 'Valide'}
sheets_p1 = pd.read_excel(SN_PHASE1, sheet_name=None, dtype=str)
sirens_valides_p1 = set()
for nom, sdf in sheets_p1.items():
    if nom in FEUILLES_VALIDES and 'nmsiren_stru' in sdf.columns:
        sirens_valides_p1.update(
            sdf['nmsiren_stru'].dropna().astype(str)
            .str.replace(r'\s', '', regex=True).str.strip()
        )
sirens_valides_p1.discard('')
print(f'SIREN validés en P1 (pour désactivation bonus) : {len(sirens_valides_p1):,}')

SIREN validés en P1 (pour désactivation bonus) : 38,483


## 1. Chargement

In [2]:
df_perimetre = pd.read_parquet(SN_PERIMETRE)
df_ul        = pd.read_parquet(SIRENE_UL_CLEAN)
df_ul['siren'] = df_ul['siren'].astype(str)

print(f'EJ à traiter Phase 2 : {len(df_perimetre):,}')
print(f'  Sous-ensembles     : {df_perimetre["sous_ensemble"].value_counts().to_dict()}')
print(f'UL SIRENE candidates  : {len(df_ul):,}')

EJ à traiter Phase 2 : 15,575
  Sous-ensembles     : {'B': 13475, 'C': 2100}
UL SIRENE candidates  : 15,111,205


## 2. Indexation des UL par commune

In [3]:
ul_par_commune = df_ul.groupby('code_commune_norm_ul', sort=False)
print(f'Communes avec au moins une UL : {ul_par_commune.ngroups:,}')

Communes avec au moins une UL : 34,808


## 3. Recherche des top 5 candidats

In [4]:
BONUS_SIREN_COHERENT = 15.0

lignes_top5 = []
lignes_orphelins = []

for _, row_ej in tqdm(df_perimetre.iterrows(), total=len(df_perimetre),
                       desc='Phase 2 sirenisation'):
    commune = row_ej['cdcommune_norm_ej']
    siren_ej = str(row_ej.get('nmsiren_stru', '') or '').strip()

    # Le bonus est désactivé pour les EJ jumeaux : leur SIREN a déjà été
    # validé en P1 par un autre EJ, donc favoriser ce SIREN en P2 reviendrait
    # à leur attribuer un match qui leur a déjà été refusé textuellement.
    bonus_applicable = (siren_ej != '') and (siren_ej not in sirens_valides_p1)

    if not commune or commune not in ul_par_commune.groups:
        lignes_orphelins.append({
            **row_ej.to_dict(),
            'siren_ref': None, 'nom_ul_retenu': None,
            'score_nom': None, 'score_adresse': None, 'score_global': None,
            'siren_coherent': False, 'bonus_applique': False,
            'score_global_ajuste': None,
            'rang': 1, 'statut_candidat': 'SANS_CANDIDAT',
        })
        continue

    candidates = ul_par_commune.get_group(commune)
    scores = []
    for _, row_ul in candidates.iterrows():
        s = scorer_paire_ej_ul(row_ej, row_ul)
        siren_ul = str(row_ul['siren']).strip()
        coherent = bool(siren_ej) and (siren_ej == siren_ul)
        bonus = BONUS_SIREN_COHERENT if (coherent and bonus_applicable) else 0.0
        score_ajuste = min(s['score_global'] + bonus, 100.0)

        scores.append({
            'siren_ref':                     row_ul['siren'],
            'denominationUniteLegale':       row_ul.get('denominationUniteLegale'),
            'sigleUniteLegale':              row_ul.get('sigleUniteLegale'),
            'adresse_siege_complete_ul':     row_ul.get('adresse_siege_complete_ul'),
            'codeCommuneEtablissement':      row_ul.get('codeCommuneEtablissement'),
            'categorieJuridiqueUniteLegale': row_ul.get('categorieJuridiqueUniteLegale'),
            'activitePrincipaleUniteLegale': row_ul.get('activitePrincipaleUniteLegale'),
            'dateCreationUniteLegale':       row_ul.get('dateCreationUniteLegale'),
            **s,
            'siren_coherent':       coherent,
            'bonus_applique':       bool(bonus > 0),
            'score_global_ajuste':  round(score_ajuste, 2),
        })

    scores.sort(key=lambda x: x['score_global_ajuste'], reverse=True)
    for rang, sc in enumerate(scores[:5], start=1):
        statut = classifier_resultat(
            sc['score_global_ajuste'], sc['score_nom'], sc['score_adresse']
        )
        lignes_top5.append({
            **row_ej.to_dict(), **sc,
            'rang': rang, 'statut_candidat': statut,
        })

df_top5 = pd.DataFrame(lignes_top5 + lignes_orphelins)

n_coherents_top1   = df_top5[(df_top5['rang'] == 1) & (df_top5['siren_coherent'])].shape[0]
n_bonus_applique   = df_top5[df_top5.get('bonus_applique', False) == True].shape[0]
n_jumeaux_protegés = (
    df_top5[(df_top5['siren_coherent']) & (df_top5.get('bonus_applique', False) == False)].shape[0]
)
print(f'\nLignes top 5 : {len(df_top5):,}')
print(f'Candidats avec SIREN cohérent au rang 1     : {n_coherents_top1:,}')
print(f'Lignes avec bonus +15 appliqué               : {n_bonus_applique:,}')
print(f'Lignes "EJ jumeaux" (cohérent mais pas bonus): {n_jumeaux_protegés:,}')
print()
print(df_top5[df_top5['rang'] == 1]['statut_candidat'].value_counts())

Phase 2 sirenisation:   0%|          | 0/15575 [00:00<?, ?it/s]


Lignes top 5 : 77,839
Candidats avec SIREN cohérent au rang 1     : 4,666
Lignes avec bonus +15 appliqué               : 5,978
Lignes "EJ jumeaux" (cohérent mais pas bonus): 15

statut_candidat
VALIDE           9800
VALIDE_FORT      2884
DOUTEUX          2776
REJETE            106
SANS_CANDIDAT       9
Name: count, dtype: int64


## 4. Aperçu

In [5]:
afficher_tableau(
    df_top5[['idstructure_stru', 'raisonsociale_stru',
             'siren_ref', 'denominationUniteLegale', 'nom_ul_retenu',
             'score_nom', 'score_adresse', 'score_global',
             'statut_candidat', 'rang']],
    'Aperçu top 5 Phase 2', max_lignes=10,
)

idstructure_stru,raisonsociale_stru,siren_ref,denominationUniteLegale,nom_ul_retenu,score_nom,score_adresse,score_global,statut_candidat,rang
1831461,C.C.A.S. DE LA ROQUE-D'ANTHERON,439043860,CYCLO CLUB DE LA ROQUE,CYCLO CLUB ROQUE,52.000000,70.000000,62.800000,VALIDE,1
1831461,C.C.A.S. DE LA ROQUE-D'ANTHERON,261301733,CENTRE COMMUNAL D'ACTION SOCIALE,CENTRE COMMUNAL ACTION SOCIALE,41.070000,37.680000,39.030000,DOUTEUX,2
1831461,C.C.A.S. DE LA ROQUE-D'ANTHERON,211300843,COMMUNE DE LA ROQUE D ANTHERON,COMMUNE ROQUE ANTHERON,67.030000,38.000000,49.610000,DOUTEUX,3
1831461,C.C.A.S. DE LA ROQUE-D'ANTHERON,533576385,SOCIETE DE CHASSE DE LA ROQUE D'ANTHERON,SOCIETE CHASSE ROQUE ANTHERON,62.590000,38.280000,48.000000,DOUTEUX,4
1831461,C.C.A.S. DE LA ROQUE-D'ANTHERON,893634279,MAISON DE LA SANTE DE LA ROQUE D'ANTHERON,MAISON SANTE ROQUE ANTHERON,60.710000,35.110000,45.350000,DOUTEUX,5
1831464,DIRECTION DES MAISONS DE L'ENFANCE,502256530,CANTINI SAINT-MICHEL,CANTINI SAINT MICHEL,30.930000,86.000000,63.970000,VALIDE,1
1831464,DIRECTION DES MAISONS DE L'ENFANCE,937502334,MARACUDJA EVENTS,MARACUDJA EVENTS,42.540000,70.840000,59.520000,VALIDE,2
1831464,DIRECTION DES MAISONS DE L'ENFANCE,844769828,POUR AINSI DIRE,AINSI DIRE,28.340000,80.000000,59.340000,VALIDE,3
1831464,DIRECTION DES MAISONS DE L'ENFANCE,841256274,MARTHINE,MARTHINE,30.910000,77.090000,58.620000,VALIDE,4
1831464,DIRECTION DES MAISONS DE L'ENFANCE,817516255,DOWNTOWN CONSULTING,DOWNTOWN CONSULTING,30.730000,76.920000,58.440000,VALIDE,5


## 5. Export Excel

In [6]:
COLS_EXPORT = [
    'idstructure_stru', 'nmfinessej_stru', 'nmfinessetab_stru', 
    'categetab_stru', 'nmsiren_stru', 'raisonsociale_stru',
    'cdape_stru', 'dtouvertstruct_stru',
    'cdcommune_stru', 'adresse_complete_ej', 'sous_ensemble',
    'siren_ref', 'siren_coherent', 'bonus_applique',
    'denominationUniteLegale', 'sigleUniteLegale',
    'nom_ul_retenu', 'adresse_siege_complete_ul', 'codeCommuneEtablissement',
    'categorieJuridiqueUniteLegale', 'activitePrincipaleUniteLegale',
    'dateCreationUniteLegale',
    'score_nom', 'score_adresse', 'score_global', 'score_global_ajuste',
    'statut_candidat', 'rang',
]

df_top5['rang'] = df_top5['rang'].astype(int)
compteurs = export_topn_excel(df_top5, SN_PHASE2, COLS_EXPORT, sheet_name='Top5')

from src.excel_export import LABELS
afficher_synthese({LABELS[s]: n for s, n in compteurs.items()},
                  'Synthèse Phase 2 sirenisation (rang 1)')
print(f'\nFichier : {SN_PHASE2}')

Statut,Nb,% du total
Valide_fort,"2,884",18.5%
Valide,"9,800",62.9%
Douteux,"2,776",17.8%
Rejeté,106,0.7%
Sans_candidat,9,0.1%
TOTAL,"15,575",100.0%



Fichier : /home/jovyan/work/projet_finess_sirene/results/sirenisation/sirenisation_phase2_top5.xlsx
